# single-cell foundation model 기초: cell embedding과 annotation

이 노트북은 single-cell RNA 발현 데이터를 foundation model의 입력으로 바꾸고, 그 출력을 cell embedding·annotation·perturbation으로 해석하는 전체 흐름을 코드로 따라갑니다. cell × gene count matrix가 어떻게 token과 embedding이 되는지, 실제 Geneformer로 PBMC 세포를 임베딩하면 무엇을 할 수 있는지, 그리고 그 결과를 어디까지 주장할 수 있는지를 한 단계씩 직접 실행하면서 확인합니다.

## 강의 목표

- cell × gene matrix, AnnData, QC, normalization, highly variable gene이 모델 입력과 어떻게 연결되는지 설명합니다.
- Geneformer의 rank-value tokenization과 scGPT의 gene+expression 입력이 무엇을 다르게 보존하는지 비교합니다.
- 실제 Geneformer를 T4 GPU에서 불러와 PBMC3k 세포 임베딩을 추출하고, PCA baseline과 비교합니다.
- marker 기반 annotation, supervised probing, batch integration, in silico perturbation, target prioritization을 수행합니다.
- UMAP 모양·marker 몇 개·embedding 거리만으로 cell type이나 생물학적 인과를 과대해석하지 않는 claim level을 익힙니다.

## 참고 자료

- **Scanpy**: Wolf et al., *Genome Biology* 2018. DOI: https://doi.org/10.1186/s13059-017-1382-0
- **UMAP**: McInnes et al., *JOSS* 2018. DOI: https://doi.org/10.21105/joss.00861
- **Geneformer**: Theodoris et al., *Nature* 2023. DOI: https://doi.org/10.1038/s41586-023-06139-9
- **scGPT**: Cui et al., *Nature Methods* 2024. DOI: https://doi.org/10.1038/s41592-024-02201-0
- **scVI**: Lopez et al., *Nature Methods* 2018. DOI: https://doi.org/10.1038/s41592-018-0229-2
- PBMC 3k public dataset: 10x Genomics / `scanpy.datasets.pbmc3k()`
- Geneformer model family: https://huggingface.co/ctheodoris/Geneformer

## 실습 범위

이 노트북의 기본 경로는 실제 PBMC3k(약 2,700개 말초혈액 단핵세포)를 scanpy로 전처리하고, Geneformer V1-10M으로 세포 임베딩을 추출하는 것입니다. 이 계산은 GPU에서 도는 것을 전제로 합니다. Colab을 쓴다면 먼저 상단 메뉴에서 **런타임 → 런타임 유형 변경 → 하드웨어 가속기를 T4 GPU로 선택**한 뒤 노트북을 처음부터 실행해 주십시오. Geneformer V1-10M은 약 10M parameter로 T4에서 가볍게 돕니다. 데이터 다운로드나 GPU가 막히면, 같은 자리에서 marker 구조를 심은 synthetic PBMC-like 데이터와 PCA 임베딩이 대신 동작하도록 설계해 두었습니다.

> **주의.** PBMC3k는 작은 공개 tutorial dataset입니다. 여기서 만드는 cell type annotation은 marker rule이나 cluster에서 나온 가설적 주석이지 정답 라벨이 아닙니다. batch는 설명을 위해 인위적으로 시뮬레이션한 것이며, 실제 donor·platform 차이가 아닙니다. 따라서 어떤 분석 결과도 disease biology나 therapeutic target 결론으로 바로 확장하지 않습니다.


## 실행 스위치 — 실제 모델 경로와 fallback

이 노트북은 시작부터 두 갈래 길을 나란히 깔아 둡니다. 어느 길로 갈지는 맨 위에 모아 둔 flag가 결정합니다. flag 값에서 `True`는 "이 경로를 시도한다", `False`는 "이 경로를 건너뛴다"는 뜻입니다.

- **기본 경로: 실제 Geneformer 임베딩.** Hugging Face에서 Geneformer V1-10M과 gene dictionary를 내려받아, PBMC 세포의 raw count를 rank-value로 토큰화하고 forward pass를 돌려 256차원 세포 임베딩을 뽑습니다. GPU 런타임을 전제로 합니다.
- **안전망: fallback 경로.** 데이터 다운로드나 모델 로드가 막히면, 같은 자리에서 marker 구조를 심은 synthetic PBMC-like 데이터의 PCA 임베딩이 대신 동작합니다. 흐름이 끊기지 않습니다.

아래 코드 셀은 노트북 전체를 움직이는 실행 스위치와 설정값을 한곳에 모읍니다. `USE_GENEFORMER`는 실제 Geneformer 임베딩을 추출할지 결정하며 기본값은 `True`입니다. `MODEL_REPO`와 `MODEL_SUBFOLDER`는 Geneformer V1-10M 모델 위치, `N_CELLS_EMBED`는 임베딩에 쓸 세포 수(전체에서 균형 있게 추려 T4에서 빠르게 끝나도록 600개로 제한), `N_NEIGHBORS`는 이웃 그래프(neighbors) 계산에 쓸 이웃 수, `RANDOM_STATE`는 재현성을 위한 seed입니다.


In [ ]:
USE_GENEFORMER = True

MODEL_REPO = "ctheodoris/Geneformer"
MODEL_SUBFOLDER = "Geneformer-V1-10M"
N_CELLS_EMBED = 600
N_NEIGHBORS = 12
RANDOM_STATE = 0

print({
    "USE_GENEFORMER": USE_GENEFORMER,
    "MODEL": f"{MODEL_REPO}/{MODEL_SUBFOLDER}",
    "N_CELLS_EMBED": N_CELLS_EMBED,
})


출력된 dictionary를 함께 읽어 봅시다. `USE_GENEFORMER`가 `True`이므로 임베딩 단계에서 실제 Geneformer를 불러와 forward pass를 돌립니다. 이 계산은 GPU에서 도는 것을 전제로 하므로, 잠시 뒤 device 설정 셀에서 GPU가 제대로 잡혔는지 먼저 확인합니다. `N_CELLS_EMBED`를 600으로 둔 것은 강의 시간 안에 임베딩이 끝나게 하기 위한 제한이며, 실제 분석에서는 전체 세포를 사용합니다. 어떤 flag를 바꿨는데 화면에 이전 값이 그대로 보인다면 이 정의 셀을 다시 실행하지 않은 것이니, 한 번 더 실행해 값을 갱신하면 됩니다.


## 실습 환경 준비

오늘 내내 쓸 패키지를 불러옵니다. 먼저 single-cell 분석의 표준 도구입니다. `scanpy`(별칭 `sc`)는 AnnData 객체로 single-cell 데이터를 다루는 핵심 라이브러리이고, `anndata`(별칭 `ad`)는 그 데이터 구조 자체를 제공합니다. Colab에 설치되어 있지 않을 수 있으므로, import가 실패하면 `pip install`로 조용히 설치한 뒤 다시 불러옵니다. 한 셀에 try/except로 묶여 있어 통째로 실행합니다.


In [ ]:
import subprocess, sys
try:
    import scanpy as sc
    import anndata as ad
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scanpy"])
    import scanpy as sc
    import anndata as ad
# Geneformer 경로에 필요한 패키지 (대개 Colab에 사전 설치됨) — 없을 때만 설치
for _pkg in ["huggingface_hub", "transformers"]:
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])
print("scanpy", sc.__version__)


다음은 수치 계산과 시각화, 그리고 sparse matrix 도구입니다. `numpy`(`np`)와 `pandas`(`pd`)는 배열과 표를, `matplotlib.pyplot`(`plt`)은 그림을 다룹니다. `scipy.sparse`(`sp`)는 single-cell count matrix가 대부분 0인 희소 행렬이라 필요하고, `IPython.display`의 `display`는 표를 보기 좋게 출력합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
from IPython.display import display


이어서 scikit-learn의 분석 도구입니다. `PCA`는 baseline 임베딩과 시각화용 차원 축소에, `train_test_split`은 reference/query 분할에, `LogisticRegression`은 supervised probing에 씁니다. `classification_report`와 `confusion_matrix`는 분류 결과 요약에, `silhouette_score`는 임베딩이 cell type이나 batch를 얼마나 분리하는지 재는 데 씁니다.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, silhouette_score


마지막으로 전역 설정을 잡습니다. `warnings.filterwarnings("ignore")`로 경고를 숨기고, scanpy의 출력 수준을 낮춰 화면을 깔끔하게 합니다. `RANDOM_STATE`로 numpy 난수 생성기 `rng`를 만들어 재현성을 확보하고, 그림 해상도를 설정합니다. 마지막 `print`가 보이면 준비가 끝난 것입니다. 여기서 오류가 나면 대개 설치 환경 문제이니, Colab이라면 런타임을 다시 시작하고 위에서부터 다시 실행하는 것이 가장 단순한 해결책입니다.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0
rng = np.random.default_rng(RANDOM_STATE)
plt.rcParams["figure.dpi"] = 120

def plot_2d(xy, labels, title):
    fig, ax = plt.subplots(figsize=(6.8, 5.2))
    for lab in pd.unique(labels):
        m = np.asarray(labels) == lab
        ax.scatter(xy[m, 0], xy[m, 1], s=10, alpha=0.7, label=str(lab))
    ax.set_title(title); ax.set_xlabel("axis 1"); ax.set_ylabel("axis 2")
    ax.legend(frameon=False, fontsize=8, markerscale=1.5)
    plt.tight_layout(); plt.show()

print("기본 패키지 import 완료")


## GPU device 잡기

실제 모델을 돌리기 전에 계산을 어디서 할지 먼저 정합니다. 아래 `get_device` 함수는 `torch.cuda.is_available()`로 GPU가 잡혔는지 확인합니다. GPU가 있으면 장치 이름과 메모리를 출력하고 `torch.device("cuda")`를 돌려줍니다. 없으면 경고와 함께 `torch.device("cpu")`를 돌려주는데, 이 경우 Geneformer forward가 느려지므로 Colab에서는 런타임을 T4 GPU로 바꾼 뒤 다시 실행하시길 권합니다. 이렇게 잡은 `device`는 뒤에서 모델과 입력을 같은 장치로 올릴 때 계속 재사용합니다.


In [ ]:
import torch

def get_device():
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f"GPU: {p.name} ({p.total_memory/1e9:.1f} GB) | torch {torch.__version__}, CUDA {torch.version.cuda}")
        return torch.device("cuda")
    print("⚠ GPU 미감지 — CPU(느림). Colab: 런타임 → 런타임 유형 변경 → T4 GPU 선택 후 재실행.")
    return torch.device("cpu")

device = get_device()


## 1. cell × gene matrix: PBMC3k 불러오기

single-cell foundation model의 입력은 cell × gene count matrix입니다. 행은 cell, 열은 gene, 값은 그 세포에서 그 gene이 몇 번 검출됐는지를 나타내는 count입니다. 여기에 batch, donor, cell type 같은 metadata가 붙습니다. 먼저 분석 전반에서 쓸 PBMC marker gene 묶음을 정의합니다. 각 cell type에서 특징적으로 높게 발현되는 gene들로, 뒤에서 cell type을 주석하고 perturbation 대상을 고를 때 기준이 됩니다.


In [ ]:
MARKER_SETS = {
    "T cell": ["CD3D", "CD3E", "IL7R", "TRAC"],
    "B cell": ["MS4A1", "CD79A", "CD79B", "BANK1"],
    "NK cell": ["NKG7", "GNLY", "KLRD1", "PRF1"],
    "Monocyte": ["LYZ", "S100A8", "S100A9", "LST1"],
    "Dendritic": ["FCER1A", "CD1C", "CLEC10A"],
    "Megakaryocyte": ["PPBP", "PF4", "ITGA2B"],
}


다운로드가 막힌 환경을 대비해, marker 구조를 심은 synthetic PBMC-like 데이터를 만드는 `make_example_pbmc_like` 함수를 먼저 정의합니다. 각 세포에 cell type을 하나 배정하고, 그 cell type의 marker gene 발현을 높인 뒤 약간의 noise를 더해 `AnnData` 객체로 돌려줍니다. 이것은 실제 데이터가 아니라 흐름을 끊기지 않게 하는 fallback이며, 실제 PBMC3k가 받아지면 쓰이지 않습니다.


In [ ]:
def make_example_pbmc_like(n_cells=720):
    genes = sorted({g for gs in MARKER_SETS.values() for g in gs}) + [f"NOISE{i:03d}" for i in range(90)]
    labels = rng.choice(list(MARKER_SETS), size=n_cells)
    gi = {g: i for i, g in enumerate(genes)}
    X = rng.normal(0.15, 0.08, size=(n_cells, len(genes)))
    for i, lab in enumerate(labels):
        for g in MARKER_SETS[lab]:
            X[i, gi[g]] += rng.normal(3.0, 0.4)
    X = np.clip(X, 0, None).astype("float32")
    obs = pd.DataFrame({"cell_type": labels}, index=[f"cell_{i:04d}" for i in range(n_cells)])
    return ad.AnnData(X=X, obs=obs, var=pd.DataFrame(index=pd.Index(genes, name="gene")))


이제 실제 PBMC3k를 불러옵니다. `sc.datasets.pbmc3k()`는 10x Genomics가 공개한 약 2,700개 말초혈액 단핵세포의 raw count matrix를 돌려줍니다. 다운로드가 실패하면 위에서 만든 synthetic 데이터로 내려가고, `is_real` flag로 어느 경로인지 기록합니다. Geneformer는 raw count를 입력으로 쓰므로, 여기서 받은 raw count를 그대로 보존하는 것이 중요합니다.


In [ ]:
try:
    adata = sc.datasets.pbmc3k()
    adata.var_names_make_unique()
    is_real = True
    print("실제 PBMC3k 로드:", adata.shape)
except Exception as exc:
    print("PBMC3k 다운로드 실패 → synthetic PBMC-like 사용:", type(exc).__name__)
    adata = make_example_pbmc_like()
    is_real = False


설명을 위해 인위적인 batch 라벨을 붙입니다. PBMC3k는 단일 donor 데이터라 실제 batch가 없으므로, 여기서는 세포를 무작위로 두 batch로 **라벨만** 나눕니다(이 단계에서 발현값 자체는 바꾸지 않습니다). 실제 batch effect는 섹션 9에서 임베딩에 인위적으로 주입해 관찰합니다. 따라서 지금 단계의 UMAP에서 batch가 고르게 섞여 보이는 것은 당연합니다(아직 효과를 주지 않았으므로). 실전에서는 이 batch 라벨을 HVG 선택 시 `batch_key`로 넘겨 batch 편향 gene이 HVG를 지배하지 않게 합니다. 마지막에 cell 수, gene 수, batch 구성을 출력합니다.

In [ ]:
adata.obs["batch"] = rng.choice(["batch_A", "batch_B"], size=adata.n_obs)
print("cells × genes:", adata.shape, "| is_real:", is_real)
display(adata.obs["batch"].value_counts().rename_axis("batch").reset_index(name="n"))


`AnnData`는 single-cell 데이터를 담는 표준 컨테이너입니다. `X`는 cell × gene 행렬, `obs`는 cell 단위 metadata(batch, cell type 등), `var`는 gene 단위 metadata, `obsm`은 PCA·embedding 같은 행렬을 보관합니다. 아래에서 구조를 출력해, 우리가 어떤 모양의 데이터를 들고 있는지 먼저 확인합니다. 실제 분석에서는 이 단계에서 gene symbol과 Ensembl ID, normalization 상태, metadata 품질을 함께 점검합니다.


In [ ]:
print(adata)
print("\nX dtype:", adata.X.dtype, "| sparse:", sp.issparse(adata.X))
display(adata.var.head())


출력에서 `AnnData object with n_obs × n_vars`가 세포 수와 gene 수입니다. 실제 PBMC3k라면 약 2,700 × 32,738 규모이고 `X`는 희소 행렬입니다. var의 index가 gene symbol(CD3D, MS4A1 등)인지 확인하는 것이 중요한데, Geneformer는 gene symbol을 Ensembl ID를 거쳐 token으로 바꾸기 때문입니다. symbol이 아니라 Ensembl ID나 다른 식별자로 되어 있으면 토큰화 단계에서 많은 gene이 빠질 수 있습니다.


## 2. QC와 전처리: baseline을 위한 정규화

Geneformer는 raw count를 입력으로 쓰지만, PCA baseline과 marker 해석을 위해서는 표준 전처리가 필요합니다. 여기서 핵심은 **raw count를 따로 보존**하는 것입니다. 먼저 raw count를 `layers["counts"]`에 복사해 두어, 뒤에서 Geneformer 토큰화에 그대로 쓸 수 있게 합니다. 그런 다음 PCA용 사본을 만들어 정규화합니다.


QC에 **mito/ribo 비율 계산과 mitochondrial 비율/유전자 수 필터(pct_counts_mt < 5%, n_genes < 2500)**를 포함합니다 - 죽어가는 세포(높은 mito%)와 유전자 수가 비정상적으로 많은 doublet 의심 세포를 거르는 표준 단계입니다(정식 doublet 검출은 Scrublet/SOLO를 씁니다). 세포 필터를 **`adata` 원본에 적용한 뒤 사본(adata_pp)을 만들어**, 뒤 단계(annotation, 임베딩)에서 세포 정렬이 어긋나지 않게 합니다.

In [ ]:
if is_real:
    adata.layers["counts"] = adata.X.copy()
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)
    adata.var["mt"]   = adata.var_names.str.upper().str.startswith("MT-")
    adata.var["ribo"] = adata.var_names.str.upper().str.startswith(("RPS", "RPL"))
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo"], percent_top=None, log1p=False, inplace=True)
    n0 = adata.n_obs
    adata = adata[(adata.obs["pct_counts_mt"] < 5) & (adata.obs["n_genes_by_counts"] < 2500)].copy()
    print(f"QC: min_genes/min_cells + mito<5% + n_genes<2500 -> {n0} -> {adata.n_obs} cells "
          f"| median mito%={adata.obs['pct_counts_mt'].median():.1f}")
adata_pp = adata.copy()
print("raw counts 보존:", "counts" in adata.layers)


실제 데이터일 때만 정규화를 적용합니다(세포/gene 필터와 mito QC는 앞 셀에서 이미 수행했습니다). `normalize_total`로 세포마다 다른 library size를 1e4로 맞춘 뒤 `log1p`로 로그 변환합니다. synthetic fallback은 이미 정리된 값이므로 이 단계를 건너뜁니다. 정규화는 high-count gene의 영향을 줄여 cell state 차이를 더 잘 보이게 합니다.

In [ ]:
if is_real:
    sc.pp.normalize_total(adata_pp, target_sum=1e4)
    sc.pp.log1p(adata_pp)
    print("정규화 후:", adata_pp.shape)
else:
    print("synthetic fallback: 정규화를 생략하고 기존 값을 사용합니다.")


highly variable gene(HVG)은 세포 간 변이가 큰 gene으로, cell state를 구분하는 신호를 주로 담습니다. `highly_variable_genes`로 상위 2,000개를 고르고, 그 gene만 남긴 사본에 `scale`을 적용해 각 gene을 평균 0, 분산 1로 맞춥니다. HVG selection은 모델이 보는 gene 목록 자체를 줄이는 단계라, 어떤 gene이 빠졌는지가 downstream 해석에 영향을 줍니다.


> 실전 팁: batch가 있는 실제 데이터에서는 `highly_variable_genes(..., batch_key="batch")`로 batch별로 HVG를 골라 batch 편향 gene이 목록을 지배하지 않게 합니다. 이 노트북의 batch는 무작위 라벨이라 여기서는 생략합니다.

In [ ]:
if is_real:
    sc.pp.highly_variable_genes(adata_pp, n_top_genes=2000, flavor="seurat")
    adata_hvg = adata_pp[:, adata_pp.var["highly_variable"]].copy()
    sc.pp.scale(adata_hvg, max_value=10)
else:
    adata_hvg = adata_pp.copy()
print("HVG matrix:", adata_hvg.shape)


이제 PCA로 차원을 줄이고 이웃 그래프와 UMAP을 계산합니다. PCA는 HVG 발현을 소수의 축으로 압축한 baseline 표현이고, neighbors와 UMAP은 그 표현 위에서 세포들의 이웃 구조를 그래프와 2차원 좌표로 만듭니다. 이 baseline은 뒤에서 Geneformer 임베딩과 비교할 기준이 됩니다. UMAP은 시각화 도구이지 cell type annotation의 정답지가 아니라는 점을 기억해야 합니다.


In [ ]:
sc.pp.pca(adata_hvg, n_comps=30, svd_solver="arpack", random_state=RANDOM_STATE)
sc.pp.neighbors(adata_hvg, n_neighbors=N_NEIGHBORS, n_pcs=30, random_state=RANDOM_STATE)
sc.tl.umap(adata_hvg, random_state=RANDOM_STATE)
print("PCA/neighbors/UMAP 완료 | X_pca:", adata_hvg.obsm["X_pca"].shape)


## 3. cell을 token으로: rank-value와 gene+expression

DNA 서열에는 좌표 순서가 있지만, cell × gene matrix의 열 순서는 파일이 정한 배열일 뿐 고정된 의미가 없습니다. 그래서 single-cell foundation model은 저마다 다른 방식으로 cell을 token sequence로 바꾸고, 그 방식이 모델의 생물학적 가정을 만듭니다. Geneformer는 한 세포 안에서 발현이 높은 gene부터 순서를 매기는 rank-value 방식을 씁니다. 절대 발현량 대신 cell 안 gene의 상대적 우선순위를 강조합니다. 먼저 raw count 행렬을 꺼내는 helper를 정의합니다.


In [ ]:
def counts_matrix():
    return adata.layers["counts"] if "counts" in adata.layers else adata.X

def dense_row(M, i):
    r = M[i]
    return np.asarray(r.todense()).ravel() if sp.issparse(r) else np.asarray(r).ravel()


한 세포에서 gene 순위를 **두 방식**으로 비교합니다. **왼쪽(raw count)**엔 B2M·ribosomal(RPS/RPL) 같은 광범위 발현 gene이 위에 오지만, **오른쪽(정규화 = `count / gene별 corpus median`)**엔 이런 housekeeping이 밀려나고 그 세포에 특이한 gene이 올라옵니다 — Geneformer가 실제로 보는 순서에 가깝습니다(구현은 cell 53). 'raw가 높은 gene' ≠ 'Geneformer 상위 gene', 이 구분이 섹션 10 perturbation 해석의 핵심입니다.

In [ ]:
ci = 0
row0 = dense_row(counts_matrix(), ci).astype(float)
syms = np.asarray(adata.var_names)
Mc_all = counts_matrix()
present = np.where(row0 > 0)[0]


**정규화값 계산.** 각 gene을 그 gene의 대표 발현(이 데이터에서 계산한 '0이 아닌 값의 median')으로 나눠 정규화값을 구합니다. 어디서나 높게 발현되는 housekeeping은 median이 커서 값이 눌려 순위가 내려갑니다.

In [ ]:
# Geneformer식 정규화의 self-contained 근사: 각 gene을 "그 gene의 0이 아닌 값들의 median"으로 나눔
Xcsc = Mc_all.tocsc() if sp.issparse(Mc_all) else np.asarray(Mc_all)
def nz_median(j):
    d = Xcsc[:, int(j)].data if sp.issparse(Xcsc) else Xcsc[:, int(j)][Xcsc[:, int(j)] > 0]
    return float(np.median(d)) if len(d) else 1.0
med = {int(j): nz_median(j) for j in present}
norm_val = np.array([row0[j] / med[int(j)] for j in present])
demo = pd.DataFrame({"gene": syms[present], "count": row0[present], "norm": np.round(norm_val, 2)})
print(f"cell: {adata.obs_names[ci]} | 검출 gene {len(present)}개 | 정규화값 계산 완료")

**두 순위 비교.** 왼쪽은 raw count 순위, 오른쪽은 정규화 순위입니다. 정규화 쪽에서 housekeeping(B2M·RPS…)이 밀려나고 그 세포에 상대적으로 특이한 gene이 위로 올라옵니다 — 오른쪽이 Geneformer가 실제로 보는 순서에 가깝습니다.

→ 이 **정규화 순위**가 섹션 5(cell 53)에서 실제 Geneformer 입력이 됩니다.

In [ ]:
raw_rank  = demo.nlargest(10, "count")[["gene", "count"]].reset_index(drop=True)
norm_rank = demo.nlargest(10, "norm")[["gene", "norm"]].reset_index(drop=True)
ranked = pd.concat([raw_rank.add_prefix("raw_"), norm_rank.add_prefix("normalized_")], axis=1)
print("cell:", adata.obs_names[ci], "| batch:", adata.obs["batch"].iloc[ci])
print("좌: raw count 순위(housekeeping/ribosomal 상위) | 우: gene별 정규화 순위 ≈ Geneformer가 보는 순서(정확한 구현은 cell 53)")
display(ranked)


scGPT 계열은 다른 방식을 씁니다. gene의 순위만 쓰는 대신, gene identity(어떤 gene인지)와 expression value(얼마나 발현됐는지)를 함께 token으로 넣고, batch나 condition 같은 token도 더합니다. 같은 세포의 상위 gene을 scGPT-style 입력 표처럼 펼치면, gene token·expression value·condition이 한 줄에 묶인 형태가 됩니다. 같은 데이터라도 tokenizer에 따라 모델이 보는 입력이 달라진다는 점이 핵심입니다.


In [ ]:
scgpt_like = pd.DataFrame({"gene": np.asarray(adata.var_names), "count": row0}).nlargest(10, "count").reset_index(drop=True)
scgpt_like["gene_token"] = scgpt_like["gene"]
scgpt_like["expression_value"] = scgpt_like["count"]
scgpt_like["condition_batch"] = adata.obs["batch"].iloc[ci]
display(scgpt_like[["gene_token", "expression_value", "condition_batch"]].head(10))


세 가지 입력 방식을 한 표로 비교합니다. 같은 cell expression vector라도 binned, rank-ordered, gene+value, gene module 중 무엇으로 바꾸느냐에 따라 모델이 보존하는 정보가 달라집니다. 모델 카드를 읽을 때 "이 모델은 cell을 어떻게 token으로 바꾸는가"를 먼저 확인하면, embedding 결과를 해석하는 기준이 잡힙니다.


In [ ]:
tok_compare = pd.DataFrame([
    {"방식": "binned expression", "대표 모델": "scBERT", "가정": "expression 값을 몇 단계 token으로 나눔"},
    {"방식": "rank-ordered genes", "대표 모델": "Geneformer", "가정": "cell 안 gene 순위가 cell state를 담음"},
    {"방식": "gene + expression", "대표 모델": "scGPT, scFoundation", "가정": "gene identity와 expression value를 함께 사용"},
    {"방식": "gene module/patch", "대표 모델": "CellPatch 등", "가정": "co-expression group으로 sequence length 축소"},
])
display(tok_compare)


## 4. Marker 기반 cell type annotation

PBMC3k에는 정답 cell type 라벨이 없습니다. 그래서 marker gene 발현으로 cell type을 추정하는 pseudo-label을 만듭니다. 각 cell type의 marker set 평균 발현을 세포마다 계산하고, 가장 높은 점수의 cell type을 붙입니다. 이것은 ground truth가 아니라 marker rule의 산물이며, marker가 빠지거나 dropout이 있으면 흔들릴 수 있다는 점을 전제로 합니다. 먼저 정규화된 발현에서 cell type별 marker score를 계산합니다.


In [ ]:
score_df = pd.DataFrame(index=adata_pp.obs_names)
for ct, markers in MARKER_SETS.items():
    present = [g for g in markers if g in adata_pp.var_names]
    if present:
        sub = adata_pp[:, present].X
        score_df[ct] = np.asarray(sub.mean(axis=1)).ravel() if sp.issparse(sub) else np.asarray(sub).mean(axis=1)
print("marker score(cell type별 marker 평균 발현) 계산:", list(score_df.columns))


세포마다 가장 높은 marker score의 cell type을 pseudo-label로 붙입니다. synthetic fallback이면 이미 cell type이 있으므로 그대로 둡니다. 그런 다음 cell type별 세포 수를 출력해, 어떤 면역세포가 많이 잡혔는지 확인합니다. PBMC에서는 보통 T cell이 가장 많고, dendritic cell이나 megakaryocyte는 드뭅니다.


> **주의 (방법의 한계).** 여기 'marker 평균 + 최고점 배정'은 가장 단순한 heuristic입니다 — 패널마다 marker 발현 스케일이 달라 점수가 완벽히 비교되지 않고, 최소 점수 문턱이 없어 애매한 세포도 강제로 한 type에 배정됩니다. 실제 분석에서는 clustering(Leiden) 후 cluster 단위로 주석하거나, 배경 보정 점수(`sc.tl.score_genes`)·자동 도구(CellTypist 등)를 씁니다. (참고: 이 데이터에서는 배경 보정+최고점 배정이 오히려 rare type을 과다 배정해, 교육용으로는 단순 평균이 더 현실적인 세포 수를 줍니다.) 이 pseudo-label이 이후 모든 단계의 기준이 되므로 그 한계를 유념하세요.

In [ ]:
if is_real and len(score_df.columns) > 0:
    adata.obs["cell_type"] = score_df.idxmax(axis=1).values
display(adata.obs["cell_type"].value_counts().rename_axis("cell_type").reset_index(name="n"))


이제 PCA/UMAP baseline 좌표 위에 annotation을 그립니다. 그림 helper `plot_2d`는 앞 import/셋업 파트(cell 11)에서 미리 정의해 두었으므로, 셀을 순서와 무관하게 다시 실행해도 안전합니다. 먼저 UMAP을 cell type 색으로 칠해, baseline 표현이 면역세포 종류를 어느 정도 나누는지 봅니다.

In [ ]:
umap_xy = adata_hvg.obsm["X_umap"]
ct_labels = adata.obs["cell_type"].to_numpy() if "cell_type" in adata.obs else adata.obs.iloc[:, 0].to_numpy()
plot_2d(umap_xy, ct_labels, "PCA/UMAP baseline colored by cell type")


같은 좌표를 batch 색으로 다시 칠합니다. cell type이 잘 나뉘어 보여도, batch가 embedding을 지배하면 cell type 구조가 batch에 의해 갈라질 수 있습니다. 그래서 같은 임베딩을 cell type 색과 batch 색으로 각각 보는 것이 중요합니다. 우리가 인위적으로 넣은 batch이므로 큰 분리는 없어야 정상이고, 만약 batch로 강하게 갈라진다면 그 신호가 어디서 왔는지 점검해야 합니다.


In [ ]:
plot_2d(umap_xy, adata.obs["batch"].to_numpy(), "PCA/UMAP baseline colored by batch")


> 해석 주의: UMAP은 visualization이지 cell type annotation의 정답지가 아닙니다. 두 세포가 UMAP에서 가까이 있다고 같은 cell type인 것도, 멀리 있다고 다른 기능인 것도 아닙니다. baseline이 이미 cell type을 잘 나눈다면, 뒤에서 볼 Geneformer 임베딩이 그 위에 무엇을 더해 주는지를 더 엄격하게 물어야 합니다. marker score가 비슷해 애매한 세포는 annotation이 흔들리는 지점이므로 따로 확인합니다.


## 잠깐 — "cell embedding"이란?

앞으로 계속 나오는 **cell embedding**을 한 번 정리하고 갑니다.

- **정의**: 한 세포의 상태를 요약한 **고정 길이 숫자 벡터**입니다(Geneformer는 256차원). 세포 하나 = 숫자 256개.
- **핵심 성질**: 이 공간에서 **가까우면 상태가 비슷한** 세포입니다. 그래서 거리·군집·분류가 의미를 가집니다.
- **PCA baseline과 무엇이 다른가** (여러분이 아는 PCA와 대비):

| | PCA (앞 섹션) | Geneformer 임베딩 |
|---|---|---|
| 학습 대상 | **이 데이터**에서 그때그때 | **3천만 세포**로 미리 학습 |
| 관계 | 선형(분산 방향 축) | 비선형(문맥 반영) |
| gene 상호작용 | 무시 | attention으로 gene–gene 문맥 학습 |

즉 PCA는 "이 데이터의 분산을 잘 보는 축", Geneformer 임베딩은 "3천만 세포에서 배운 사전지식으로 이 세포를 요약한 벡터"입니다. 아래에서 실제로 뽑아 PCA와 비교합니다.

## 5. 실제 Geneformer 세포 임베딩

baseline을 세웠으니 실제 foundation model 임베딩으로 넘어갑니다. Geneformer는 약 30M개 human single-cell transcriptome으로 학습한 모델로, 세포의 rank-ordered gene sequence를 받아 256차원 임베딩을 만듭니다. 전체 세포를 다 임베딩하면 시간이 걸리므로, cell type별로 균형 있게 `N_CELLS_EMBED`개를 추려 사용합니다. 먼저 임베딩에 쓸 세포를 고르고, 각 세포의 raw count 행과 gene symbol 목록을 준비합니다.


> `N_CELLS_EMBED`(=600)는 상한입니다. cell type별로 고르게 추리므로, 희귀 세포(Dendritic, Megakaryocyte)가 타입당 몫보다 적으면 실제 임베딩 세포 수는 600보다 작을 수 있습니다(출력의 '임베딩 대상 세포' 수 확인).

In [ ]:
ct_arr = adata.obs["cell_type"].to_numpy()
n_each = max(1, N_CELLS_EMBED // len(pd.unique(ct_arr)))
sub_idx = []
for ct in pd.unique(ct_arr):
    pos = np.where(ct_arr == ct)[0]
    sub_idx.extend(rng.choice(pos, size=min(len(pos), n_each), replace=False))
sub_idx = np.array(sorted(sub_idx))
sub_obs = adata.obs.iloc[sub_idx].reset_index(drop=True)
symbols = np.asarray(adata.var_names)
Mc = counts_matrix()
count_rows = [dense_row(Mc, i) for i in sub_idx]
print("임베딩 대상 세포:", len(sub_idx))


아래 셀이 기본 경로, 즉 실제 Geneformer로 임베딩을 뽑는 부분입니다. `if`와 그 안의 `try`/`except`가 한 덩어리로 맞물려 있어 한 셀로 둡니다. Hugging Face에서 gene dictionary 세 개와 Geneformer V1-10M 모델을 내려받습니다. `token_dict`는 Ensembl ID를 token id로, `median_dict`는 gene별 정규화 기준값을, `name_id`는 gene symbol을 Ensembl ID로 바꾸는 사전입니다. `tokenize_cell`은 한 세포의 count를 받아, 발현이 있는 gene을 median으로 나눠 normalized rank를 매기고 상위 gene의 token id를 순서대로 돌려줍니다. `geneformer_embed`는 이 token sequence를 batch로 묶어 모델에 넣고, last hidden state를 attention mask로 mean pooling해 세포당 256차원 벡터를 만듭니다. 어느 단계에서든 실패하면 `except`가 메시지를 찍고 다음 셀의 fallback이 이어받습니다.


> **실행 안내.** 이 셀은 모델과 dictionary 다운로드, forward pass 때문에 **수 분 걸릴 수 있고**, 진행 막대가 잠깐 멈춘 듯 보여도 정상입니다. 로드 중 `pooler weights ... newly initialized / Consider training` 경고가 뜨는데, 우리는 pooler 대신 `last_hidden_state`를 직접 mean-pooling하므로 **무시해도 됩니다**. 참고로 *embedding*은 '세포를 거리가 의미를 갖는 고정 길이 벡터로 바꾼 것', *mean-pooling*은 '토큰별 벡터를 평균 내 세포 하나의 벡터로 요약'하는 것입니다. 다운로드가 막히면 다음 셀에서 자동으로 PCA fallback으로 넘어가니, 출력의 `embedding source`가 'Geneformer ...'인지 'fallback ...'인지 꼭 확인하세요(경로에 따라 이후 숫자가 달라집니다).

### 아래 셀은 4단계를 한 셀에 합칩니다

길어 보이지만 하는 일은 4단계입니다. 코드를 다 읽을 필요는 없고 **흐름만** 잡으세요. (모델·데이터 다운로드로 **수 분** 걸리고, 진행 막대가 도는 건 정상입니다.)

1. **사전·모델 로드** — Hugging Face에서 gene dictionary 3개(token / median / name→id)와 Geneformer V1-10M을 내려받습니다.
2. **토큰화** — 각 세포의 raw count를 **섹션 3에서 본 그 정규화 순위**(`count / gene median`)로 바꿔, 상위 gene의 token id 시퀀스를 만듭니다. ← *섹션 3 데모가 바로 여기서 실제로 쓰입니다.*
3. **forward pass** — token 시퀀스를 모델에 넣어 `last_hidden_state`를 얻습니다. 이건 **token(gene)마다 하나씩 나오는 문맥 벡터**입니다(한 세포 = 여러 gene 벡터).
4. **mean-pooling** — 그 gene 벡터들을 (padding 제외) **평균 내어 세포당 벡터 1개**(256차원)로 만듭니다. 이게 위에서 정의한 cell embedding입니다.

다운로드나 GPU가 막히면 `except`가 받아, 다음 셀의 fallback(합성 데이터 + PCA)으로 자연스럽게 넘어갑니다.

In [ ]:
cell_emb = None
embedding_source = None
if USE_GENEFORMER:
    try:
        import pickle
        from huggingface_hub import hf_hub_download
        from transformers import AutoModel
        def _load_pkl(fn):
            return pickle.load(open(hf_hub_download(MODEL_REPO, fn), "rb"))
        token_dict  = _load_pkl("geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl")
        median_dict = _load_pkl("geneformer/gene_dictionaries_30m/gene_median_dictionary_gc30M.pkl")
        name_id     = _load_pkl("geneformer/gene_dictionaries_30m/gene_name_id_dict_gc30M.pkl")
        gf_model = AutoModel.from_pretrained(MODEL_REPO, subfolder=MODEL_SUBFOLDER).to(device).eval()

        def tokenize_cell(counts, syms, max_len=2048):
            toks, vals = [], []
            for sym, v in zip(syms, counts):
                if v <= 0:
                    continue
                ens = name_id.get(sym)
                if ens is None or ens not in median_dict or ens not in token_dict:
                    continue
                toks.append(token_dict[ens]); vals.append(v / median_dict[ens])
            order = np.argsort(vals)[::-1][:max_len]
            return [toks[i] for i in order]

        def geneformer_embed(rows, syms, batch_size=16):
            pad = token_dict.get("<pad>", 0); out = []
            ids_all = [tokenize_cell(r, syms) or [pad] for r in rows]
            for s in range(0, len(ids_all), batch_size):
                chunk = ids_all[s:s+batch_size]; ml = max(len(x) for x in chunk)
                ids = torch.tensor([x + [pad]*(ml-len(x)) for x in chunk], device=device)
                attn = (ids != pad).long()
                with torch.no_grad():
                    h = gf_model(input_ids=ids, attention_mask=attn).last_hidden_state
                m = attn.unsqueeze(-1).to(h.dtype)
                out.append(((h*m).sum(1)/m.sum(1).clamp(min=1)).float().cpu().numpy())
            return np.concatenate(out, 0)

        cell_emb = geneformer_embed(count_rows, symbols)
        embedding_source = "Geneformer V1-10M mean-pooled last hidden state"
        print(f"Geneformer embedding: {cell_emb.shape} | device: {device}")
        if device.type == "cuda":
            print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    except Exception as exc:
        print("Geneformer embedding 실패. fallback으로 전환합니다.")
        print(type(exc).__name__, exc)


다음은 fallback 경로입니다. `cell_emb`가 아직 `None`이면(Geneformer를 끄거나 로드가 실패한 경우) 앞에서 계산한 PCA 좌표를 임베딩으로 씁니다. 추려 둔 세포의 PCA representation을 가져와 같은 분석을 이어 갑니다. 이렇게 하면 모델 다운로드가 막혀도 시각화와 nearest neighbor 흐름이 끊기지 않습니다.


In [ ]:
if cell_emb is None:
    cell_emb = np.asarray(adata_hvg.obsm["X_pca"])[sub_idx]
    embedding_source = "fallback: PCA(30) baseline"
print("embedding source:", embedding_source, "| shape:", cell_emb.shape)


## 6. 세포 임베딩 시각화

임베딩을 2차원으로 투영해 그립니다. Geneformer 임베딩은 256차원이므로 PCA로 2차원으로 줄여 산점도를 그립니다. cell type 색으로 칠해, 모델 임베딩이 면역세포 종류를 어떻게 배치하는지 봅니다. baseline UMAP과 비교하면, 같은 세포를 두 표현이 어떻게 다르게 보는지 감을 잡을 수 있습니다.


In [ ]:
emb_xy = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(cell_emb)
plot_2d(emb_xy, sub_obs["cell_type"].to_numpy(), "cell embedding (2D projection) by cell type")


같은 임베딩을 batch 색으로 다시 칠합니다. cell type이 잘 나뉘는 임베딩이라도 batch가 섞이지 않고 따로 뭉치면, 그 임베딩에는 생물학적 신호와 technical 신호가 섞여 있는 것입니다. 우리가 인위적으로 넣은 batch이므로 cell type 안에서 batch가 고르게 섞여 보여야 정상입니다. 이 비교가 다음 integration 단계의 출발점이 됩니다.


In [ ]:
plot_2d(emb_xy, sub_obs["batch"].to_numpy(), "cell embedding colored by batch")


## 7. 비슷한 세포 찾기

임베딩 공간에서 각 세포와 가장 닮은 이웃을 찾아봅니다. `cosine_similarity`로 모든 세포 쌍의 유사도를 계산하고, 자기 자신은 제외하기 위해 대각선을 음의 무한대로 채웁니다. 그런 다음 각 세포의 가장 가까운 이웃을 찾아 query 세포와 nearest 세포의 cell type을 나란히 둡니다. 같은 cell type끼리 가까운지, 아니면 다른 type이 끼어드는지를 보면 임베딩이 무엇을 보존하는지 알 수 있습니다.


> *cosine similarity*는 두 벡터가 이루는 각도 기반 유사도로, 1이면 같은 방향, 0이면 직교, -1이면 반대입니다. 여기서는 임베딩 벡터 사이 유사도로 씁니다.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(cell_emb)
np.fill_diagonal(sim, -np.inf)
nn = sim.argmax(axis=1)
neighbor_df = pd.DataFrame({
    "query_type": sub_obs["cell_type"].to_numpy(),
    "nearest_type": sub_obs["cell_type"].to_numpy()[nn],
    "cosine_similarity": np.round(sim[np.arange(len(nn)), nn], 3),
})
agree = float((neighbor_df["query_type"] == neighbor_df["nearest_type"]).mean())
print(f"nearest neighbor가 같은 cell type인 비율: {agree:.2f}")
disagree = neighbor_df[neighbor_df["query_type"] != neighbor_df["nearest_type"]]
print(f"다른 cell type이 가장 가까운 세포: {len(disagree)}개 (아래는 그 예시 - 여기서 배울 게 많습니다)")
display(disagree.head(8) if len(disagree) else neighbor_df.head(8))


nearest neighbor가 같은 cell type인 비율이 높을수록, 임베딩이 cell type 구조를 잘 보존한다는 뜻입니다. 다만 이 "가까움"은 임베딩이 학습한 표현 안에서의 유사성이지 기능적·인과적 동일성이 아닙니다. 다른 cell type이 가까운 행이 보이면, marker가 겹치거나 doublet, 혹은 중간 상태일 수 있으므로 그 세포를 따로 확인합니다. 실제 분석에서 이 표는 결론이 아니라 "어떤 세포를 함께 들여다볼지" 정하는 출발점입니다.


> 위 표는 일부러 **다른 cell type이 가장 가까운(불일치) 세포**만 보여 줍니다 - 잘 맞는 세포보다 여기서 배울 게 많기 때문입니다(예: T와 NK 혼동).

## 8. Supervised probing: 임베딩에 정보가 있는가

probing은 임베딩을 고정한 채 그 위에 작은 분류기만 학습해, 임베딩이 cell type을 구분할 정보를 담고 있는지 평가하는 방법입니다. 세포를 reference(train)와 query(test)로 나누고, reference 임베딩으로 logistic regression을 학습한 뒤 query에서 cell type을 얼마나 맞히는지 봅니다. 먼저 cell type 비율을 유지한 채 데이터를 나눕니다.


> **split 주의.** 여기서는 시간상 가장 단순한 **무작위 cell split**을 씁니다 - 같은 donor/batch의 비슷한 세포가 train/test 양쪽에 들어갈 수 있어 가장 쉬운 평가입니다(슬라이드 p.40, 그리고 이 노트북 마무리 checklist가 강조하는 donor/batch/tissue holdout이 실제로는 필요합니다). 또한 label(pseudo-label)이 marker 발현에서 나왔고 임베딩도 같은 발현에서 나오므로 이 검증은 부분적으로 **순환적**일 수 있습니다. 따라서 아래 점수는 '임베딩에 cell type 정보가 있다' 정도로만 읽고 일반화 성능으로 과대해석하지 않습니다.

In [ ]:
y = sub_obs["cell_type"].to_numpy()
Xtr, Xte, ytr, yte = train_test_split(cell_emb, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)
print("reference:", Xtr.shape[0], "| query:", Xte.shape[0])


reference 임베딩으로 logistic regression probe를 학습합니다. `class_weight="balanced"`는 cell type 불균형(T cell이 많고 dendritic이 적음)을 보정합니다. 학습한 probe로 query를 예측하고 `classification_report`로 cell type별 precision·recall·f1을 봅니다. 점수가 높으면 임베딩이 cell type 정보를 잘 담은 것이지만, 이것이 생물학적 정확성을 증명하지는 않습니다 — pseudo-label 자체가 marker rule의 산물이기 때문입니다.


> 아래 리포트에서 각 행은 cell type별 precision/recall/f1이고 `support`는 그 type의 test 세포 수입니다. **support가 작은 type(예: Megakaryocyte n이 한 자리)의 완벽 점수는 통계적으로 불안정**하니(세포 하나만 틀려도 크게 흔들림) 소수 둘째 자리 숫자를 과신하지 마세요.

In [ ]:
probe = LogisticRegression(max_iter=2000, class_weight="balanced")
probe.fit(Xtr, ytr)
pred = probe.predict(Xte)
print(classification_report(yte, pred, zero_division=0))


혼동 행렬로 어떤 cell type을 어디로 헷갈렸는지 봅니다. 대각선이 맞게 분류한 경우이고, 대각선 밖 숫자가 혼동입니다. 면역세포 중 marker가 일부 겹치는 종류(예: T cell과 NK cell)가 서로 헷갈리는지, 드문 cell type이 흔한 type으로 흡수되는지를 보는 것이 점수 하나보다 유용합니다.


In [ ]:
labels_sorted = sorted(pd.unique(y))
cm = confusion_matrix(yte, pred, labels=labels_sorted)
display(pd.DataFrame(cm, index=[f"true {x}" for x in labels_sorted], columns=[f"pred {x}" for x in labels_sorted]))


## 9. Batch integration — 진짜 두 데이터셋으로

지금까지는 pbmc3k **한 데이터셋**에 무작위 batch 라벨을 붙였을 뿐이라, 섹션 6에서 batch가 그냥 섞여 보였습니다(효과가 없었으니 당연). 이제 **진짜 두 번째 데이터셋**을 가져옵니다: **pbmc_1k_v3**(10x 3' **v3**) vs pbmc3k(10x 3' **v1**) — 같은 PBMC지만 chemistry가 달라 **진짜 technical batch effect**가 생깁니다.

핵심 질문: **① PCA baseline에서 batch가 갈라지는가 → ② Geneformer zero-shot 임베딩이 batch를 섞어주는가 → ③ Harmony·scVI 같은 전용 도구와 비교하면?**

> 정직한 예고: Geneformer의 rank 인코딩은 depth 차이에 강건해서 v1/v3를 **부분적으로는** 섞어줄 수 있지만, zero-shot FM이 batch를 완전히 없앤다는 보장은 없습니다(벤치마크에서 자주 남습니다). 결과가 어떻든 — "얼마나 (못) 지우나" — 그 자체가 교육 포인트입니다.

In [ ]:
# 두 번째 batch 데이터셋(pbmc_1k_v3, 10x v3) 다운로드 + QC + 주석
import os, urllib.request
import anndata as ad
_URL = "https://cf.10xgenomics.com/samples/cell-exp/3.0.0/pbmc_1k_v3/pbmc_1k_v3_filtered_feature_bc_matrix.h5"
_FN = "pbmc_1k_v3.h5"
def _annotate(a):
    pp = a.copy(); sc.pp.normalize_total(pp, target_sum=1e4); sc.pp.log1p(pp)
    sco = pd.DataFrame(index=a.obs_names)
    for ct, ms in MARKER_SETS.items():
        pr = [g for g in ms if g in pp.var_names]
        if pr:
            X = pp[:, pr].X
            sco[ct] = np.asarray(X.mean(axis=1)).ravel()
    a.obs["cell_type"] = sco.idxmax(axis=1).values
    return a
_brng = np.random.default_rng(RANDOM_STATE)   # 독립 rng (global rng 안 건드림 → 뒤 perturbation 결과 불변)
def _subs(a, n):
    idx = _brng.choice(a.n_obs, size=min(n, a.n_obs), replace=False)
    return a[idx].copy()

HAS_BATCH2 = False
try:
    if not os.path.exists(_FN):
        print("pbmc_1k_v3 다운로드 중 (~5MB)...")
        # 10x Cloudflare CDN은 기본 urllib UA를 403으로 막으므로 브라우저 UA 헤더 필요
        _req = urllib.request.Request(_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(_req, timeout=120) as _r, open(_FN, "wb") as _f:
            _f.write(_r.read())
    b2 = sc.read_10x_h5(_FN); b2.var_names_make_unique()
    sc.pp.filter_cells(b2, min_genes=200); sc.pp.filter_genes(b2, min_cells=3)
    b2.layers["counts"] = b2.X.copy(); _annotate(b2)

    b1 = adata.copy()                     # pbmc3k (raw counts in layers["counts"], gene_ids in var)
    if "cell_type" not in b1.obs: _annotate(b1)
    b1.obs["batch"] = "10x_v1"; b2.obs["batch"] = "10x_v3"
    N_PER = 350
    s1, s2 = _subs(b1, N_PER), _subs(b2, N_PER)
    # Geneformer는 세포별 독립 토큰화 → 배치가 유전자를 공유할 필요 없음. symbol 보존.
    s1.var["symbol"] = s1.var_names.astype(str); s2.var["symbol"] = s2.var_names.astype(str)
    gf1 = (s1.layers["counts"], np.asarray(s1.var["symbol"]))
    gf2 = (s2.layers["counts"], np.asarray(s2.var["symbol"]))
    # PCA/Harmony/scVI baseline은 Ensembl ID로 join (pbmc3k=hg19, v3=GRCh38 → symbol 말고 gene_ids)
    for a in (s1, s2):
        a.var_names = a.var["gene_ids"].astype(str).values; a.var_names_make_unique()
    comb = ad.concat([s1, s2], join="inner")
    comb.layers["counts"] = comb.X.copy()
    HAS_BATCH2 = True
    print("combined:", comb.shape, "| batch:", comb.obs["batch"].value_counts().to_dict())
    display(pd.crosstab(comb.obs["batch"], comb.obs["cell_type"]))
except Exception as e:
    print("batch2 로드 실패 — batch integration 데모를 건너뜁니다:", type(e).__name__, e)


먼저 **baseline**: 두 데이터셋을 합쳐 표준 전처리(정규화·HVG·PCA)한 뒤, 같은 좌표를 **batch 색**과 **cell type 색**으로 봅니다. batch가 갈라져 보이면 실제 batch effect가 있는 것입니다.

In [ ]:
# baseline: 합친 데이터에 표준 PCA → batch / cell type 색으로 시각화
def _sil(X, key):
    from sklearn.metrics import silhouette_score as _ss
    return round(float(_ss(X, comb.obs[key])), 3)
def _pca2(X):
    return PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)

results = {}
if HAS_BATCH2:
    cp = comb.copy(); sc.pp.normalize_total(cp, target_sum=1e4); sc.pp.log1p(cp)
    sc.pp.highly_variable_genes(cp, n_top_genes=2000)
    cp = cp[:, cp.var.highly_variable].copy(); sc.pp.scale(cp, max_value=10)
    X_pca = PCA(n_components=30, random_state=RANDOM_STATE).fit_transform(cp.X)
    results["PCA baseline"] = X_pca
    xy = _pca2(X_pca)
    plot_2d(xy, comb.obs["batch"].to_numpy(), "PCA baseline — colored by BATCH (separated = batch effect)")
    plot_2d(xy, comb.obs["cell_type"].to_numpy(), "PCA baseline — colored by cell type")
    print(f"batch_silhouette={_sil(X_pca,'batch')} (낮을수록 섞임)  celltype_silhouette={_sil(X_pca,'cell_type')}")


이제 **Geneformer zero-shot 임베딩**입니다. 두 batch를 각각 독립적으로 토큰화·임베딩(같은 Geneformer vocab)한 뒤 합쳐서, batch 색으로 봅니다. baseline보다 batch가 **더 섞이면** Geneformer가 batch에 어느 정도 강건하다는 뜻입니다. (fallback 경로에선 실제 Geneformer가 없어 baseline과 같게 표시됩니다 — 실제 Geneformer는 Colab에서.)

In [ ]:
# Geneformer 임베딩 (배치별 독립) → batch 색
if HAS_BATCH2:
    if str(embedding_source).startswith("Geneformer"):
        gf_emb = np.vstack([geneformer_embed(list(gf1[0].toarray() if hasattr(gf1[0], "toarray") else gf1[0]), gf1[1]),
                            geneformer_embed(list(gf2[0].toarray() if hasattr(gf2[0], "toarray") else gf2[0]), gf2[1])])
        _tag = "Geneformer (zero-shot)"
    else:
        gf_emb = results["PCA baseline"]   # fallback: 실제 Geneformer 없음 → baseline로 대체 표시
        _tag = "Geneformer (fallback=baseline)"
    results["Geneformer"] = gf_emb
    plot_2d(_pca2(gf_emb), comb.obs["batch"].to_numpy(), f"{_tag} — colored by BATCH")
    print(f"[{_tag}] batch_silhouette={_sil(gf_emb,'batch')}  celltype_silhouette={_sil(gf_emb,'cell_type')}")


마지막으로 **전용 integration 도구** Harmony(선형 보정)와 scVI(생성 모델)를 같은 데이터에 적용해, batch를 섞으면서 cell type을 보존하는지 비교합니다. scVI는 무거워서(설치+학습) `RUN_SCVI` 플래그로 끌 수 있습니다.

In [ ]:
# Harmony + scVI (전용 integration 도구) — 없으면 설치, 실패해도 계속
RUN_SCVI = True
if HAS_BATCH2:
    try:
        try:
            import harmonypy
        except ImportError:
            import subprocess, sys; subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "harmonypy"]); import harmonypy
        _ho = harmonypy.run_harmony(results["PCA baseline"], comb.obs, ["batch"])
        _Z = np.asarray(_ho.Z_corr); results["Harmony"] = _Z if _Z.shape[0] == comb.n_obs else _Z.T
        plot_2d(_pca2(results["Harmony"]), comb.obs["batch"].to_numpy(), "Harmony — colored by BATCH")
        print(f"[Harmony] batch_silhouette={_sil(results['Harmony'],'batch')}  celltype_silhouette={_sil(results['Harmony'],'cell_type')}")
    except Exception as e:
        print("Harmony 건너뜀:", type(e).__name__, e)
    if RUN_SCVI:
        try:
            try:
                import scvi
            except ImportError:
                import subprocess, sys; subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scvi-tools"]); import scvi
            _cv = comb.copy(); _cv.X = _cv.layers["counts"].copy()
            scvi.model.SCVI.setup_anndata(_cv, batch_key="batch")
            _m = scvi.model.SCVI(_cv, n_latent=30); _m.train(max_epochs=100, enable_progress_bar=False)
            results["scVI"] = _m.get_latent_representation()
            plot_2d(_pca2(results["scVI"]), comb.obs["batch"].to_numpy(), "scVI — colored by BATCH")
            print(f"[scVI] batch_silhouette={_sil(results['scVI'],'batch')}  celltype_silhouette={_sil(results['scVI'],'cell_type')}")
        except Exception as e:
            print("scVI 건너뜀:", type(e).__name__, e)


네 방법을 한 표로 비교합니다. **batch_silhouette은 낮을수록**(batch가 섞일수록) 좋고, **celltype_silhouette은 baseline 수준을 유지**해야(생물학 보존) 좋습니다. batch만 낮추고 celltype이 무너지면 over-integration입니다.

In [ ]:
# 요약 표: 각 방법의 batch / cell type silhouette
if HAS_BATCH2 and results:
    tab = pd.DataFrame({k: {"batch_sil (↓좋음)": _sil(v, "batch"), "celltype_sil (유지좋음)": _sil(v, "cell_type")}
                        for k, v in results.items()}).T
    display(tab)


정리 — 이 실습에서 확인한 것:
- **baseline**: 실제 v1/v3 batch가 있다(batch_sil > 0). 정규화가 depth 차이를 잡아줘 생각보다 크지 않을 수 있음.
- **Geneformer zero-shot**: batch를 얼마나 섞는가? — rank 인코딩 덕에 부분적으로 강건하나, 완전 통합은 미보장.
- **Harmony / scVI**: batch를 더 낮추며 cell type을 보존 → **전용 도구가 왜 여전히 필요한가**를 보여줌.

교훈: **foundation model 임베딩도 zero-shot으론 batch를 완전히 없애지 못할 수 있으니, batch가 있는 실제 데이터에선 integration 지표(batch+biology 둘 다)를 반드시 함께 확인해야 합니다.**

### in-silico perturbation이 하는 일 (직관)

한 세포의 입력에서 어떤 gene(들)을 **지우고**, 다시 임베딩을 뽑아, **원래 임베딩에서 얼마나 멀어졌는지**(1 − cosine)를 잽니다.

- 많이 움직임 = 모델이 그 세포를 표현할 때 그 gene에 **많이 의존**한다는 뜻 → `effect_shift`가 큼.
- 거의 안 움직임 = 그 gene이 그 세포 표현에 별로 중요치 않음.

이건 **인과가 아니라** "모델 표현 안에서 그 program이 빠지면 얼마나 달라 보이나"입니다(claim level 주의). 그리고 뒤에서 보듯 **`effect_shift` 하나로는 부족**합니다 — **크게·넓게 발현되는 gene이 무조건 크게 나오기** 때문에, **specificity**(한 cell type에 집중되는 정도)를 반드시 함께 봐야 합니다. 이 **두 축(effect × specificity)**이 이 섹션과 다음 우선순위화의 핵심입니다.

## 10. In silico perturbation

Geneformer는 한 세포의 rank sequence에서 특정 gene token을 빼는 방식으로 perturbation을 모사합니다. 그런데 gene 하나만 빼면 세포당 수백 개 token 중 하나가 빠지는 셈이라 임베딩이 거의 움직이지 않습니다. 그래서 여기서는 한 cell type을 정의하는 **marker program 전체(여러 gene)**를 함께 제거해, cell state 수준의 변화를 봅니다. program을 제거한 세포의 임베딩이 원래에서 얼마나 멀어지는지를 보면, 그 program이 cell state 표현에 얼마나 기여하는지 가늠할 수 있습니다. Geneformer 임베딩이 성공했으면 토큰 제거 후 실제 재임베딩으로, fallback이면 expression 공간에서 해당 gene들을 0으로 한 벡터의 변화로 계산합니다.


> 참고: Geneformer의 정식 in-silico perturbation은 보통 **단일 gene** 토큰을 지우거나(deletion) 맨 앞으로 옮겨(activation) 효과를 봅니다. 여기서는 한 gene만 지우면 변화가 너무 작아, 교육용으로 **marker program 전체(여러 gene)**를 함께 지워 cell-state 수준의 변화를 봅니다.

In [ ]:
GENEFORMER_OK = embedding_source.startswith("Geneformer")

# specificity는 정규화(log1p) 발현으로 계산 - annotation(cell 42)과 같은 layer, library size 보정
def gene_mean_by_celltype(gene):
    if gene not in adata_pp.var_names:
        return None
    col = adata_pp[:, gene].X
    col = np.asarray(col.todense()).ravel() if sp.issparse(col) else np.asarray(col).ravel()
    return pd.Series(col).groupby(ct_arr).mean()

print("perturbation 경로:", "Geneformer 토큰 제거" if GENEFORMER_OK else "expression proxy")


`perturb_shift` 함수는 program의 marker gene 목록을 받아, 그 중 하나라도 발현하는 세포를 골라 program의 모든 gene token을 제거한 뒤 임베딩 이동(1 − cosine similarity)을 평균으로 돌려줍니다. Geneformer 경로에서는 token sequence에서 해당 gene들을 빼고 다시 임베딩하고, fallback에서는 log-normalized expression vector에서 그 gene들을 0으로 만든 차이를 봅니다. 발현하는 세포가 없으면 측정할 수 없으므로 결측으로 둡니다. 이 값은 인과적 효과가 아니라 모델 표현 안에서 "그 program이 빠지면 얼마나 달라 보이는가"입니다.


In [ ]:
def perturb_shift(genes, n=24):
    gjs = [int(np.where(symbols == g)[0][0]) for g in genes if len(np.where(symbols == g)[0])]
    if not gjs:
        return np.nan, 0
    cand = [i for i in sub_idx if any(dense_row(Mc, i)[gj] > 0 for gj in gjs)]
    if not cand:
        return np.nan, 0
    cells = list(rng.choice(cand, size=min(len(cand), n), replace=False))
    rows = [dense_row(Mc, i).astype(float) for i in cells]
    rows_ko = [r.copy() for r in rows]
    for r in rows_ko:
        for gj in gjs:
            r[gj] = 0.0
    if GENEFORMER_OK:
        base, pert = geneformer_embed(rows, symbols), geneformer_embed(rows_ko, symbols)
    else:
        def ev(r):
            s = r.sum(); return np.log1p(r / (s if s > 0 else 1) * 1e4)
        base = np.array([ev(r) for r in rows]); pert = np.array([ev(r) for r in rows_ko])
    cos = (base * pert).sum(1) / (np.linalg.norm(base, axis=1) * np.linalg.norm(pert, axis=1) + 1e-8)
    return float(np.mean(1.0 - cos)), len(cells)


교육용 program 후보 표를 만듭니다. 각 후보는 cell type program(그 cell type을 정의하는 marker gene set)과 prior plausibility를 나타내는 literature support로 구성됩니다. 마지막 housekeeping(GAPDH·ACTB·B2M)은 특정 cell state에 특이적이지 않은 negative control입니다. `program_genes`는 program 이름을 marker gene 목록으로 바꿔 주고, 각 program에 대해 `perturb_shift`로 effect를 계산합니다.


> `literature_support` 값(0.2~0.9)은 이 실습을 위해 **임의로 정한 예시 수치**이며 실제 evidence(Open Targets, GWAS 등)에서 온 것이 아닙니다. 측정값(effect_shift, specificity)과 성격이 다른 '가상의 prior'임을 유념하세요.

In [ ]:
HK_GENES = ["GAPDH", "ACTB", "B2M"]
def program_genes(p):
    return MARKER_SETS.get(p, HK_GENES)

targets = pd.DataFrame([
    {"program": "T cell", "literature_support": 0.9},
    {"program": "B cell", "literature_support": 0.8},
    {"program": "Monocyte", "literature_support": 0.7},
    {"program": "NK cell", "literature_support": 0.6},
    {"program": "housekeeping", "literature_support": 0.2},
])
eff = [perturb_shift(program_genes(p)) for p in targets["program"]]
targets["effect_shift"] = [round(e[0], 4) if not np.isnan(e[0]) else np.nan for e in eff]
targets["n_cells"] = [e[1] for e in eff]
targets["genes"] = [", ".join(program_genes(p)) for p in targets["program"]]
display(targets[["program", "genes", "effect_shift", "n_cells", "literature_support"]])


### 이동을 눈으로 — 화살표로 보는 perturbation (모든 cell type)

`effect_shift`가 숫자라 감이 안 오니 임베딩 위에서 직접 봅니다. **각 cell type program마다** 그 program을 발현하는 세포가 유전자 제거 후 **얼마나 이동하는지**를 화살표로 그립니다(세포가 너무 적은 type은 자동 생략).

- **왼쪽 — 단일 gene 하나만 제거**: 화살표가 거의 안 보입니다(제자리).
- **오른쪽 — program 전체 제거**: 화살표가 길어집니다(뚜렷이 이동).

여러 type을 훑어보고 대비가 가장 뚜렷한 것을 고르면 됩니다. 이게 노트북이 단일 gene이 아니라 program 단위로 지우는 이유입니다.

> ⚠️ 2D 화살표 길이는 256차원 이동을 평면에 투영한 **그림자**입니다 — 실제 크기는 위 표의 cosine(`effect_shift`) 값이 정확합니다. fallback 경로에선 임베딩이 log-발현이라 이동이 발현량 크기 아티팩트입니다(**실제 Geneformer일 때 가장 의미 있음**).

In [ ]:
# in-silico perturbation을 임베딩 위에서 시각화: 모든 cell type program을 각각 (단일 gene vs program 전체)
def _embed_rows(rows):
    if str(embedding_source).startswith("Geneformer"):
        return geneformer_embed(rows, symbols)
    def ev(r):
        t = r.sum(); return np.log1p(r / (t if t > 0 else 1) * 1e4)
    return np.array([ev(r) for r in rows])
def _idx(g):
    hit = np.where(symbols == g)[0]
    return int(hit[0]) if len(hit) else None
def _ko(rows, genes):
    gj = [j for j in (_idx(g) for g in genes) if j is not None]
    out = [r.copy() for r in rows]
    for r in out:
        for j in gj:
            r[j] = 0.0
    return out
def _mcos(a, b):
    return float(np.mean(1 - (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-8)))

def viz_program(prog, n=40, min_cells=8):
    genes = program_genes(prog); single = genes[0]
    gj_all = [j for j in (_idx(g) for g in genes) if j is not None]
    cand = [i for i in sub_idx if any(dense_row(Mc, i)[j] > 0 for j in gj_all)]
    if len(cand) < min_cells:
        print(f"[{prog}] 발현 세포 {len(cand)}개 — 너무 적어 생략"); return
    cells = list(rng.choice(cand, size=min(len(cand), n), replace=False))
    rows = [dense_row(Mc, i).astype(float) for i in cells]
    base = _embed_rows(rows); ko_s = _embed_rows(_ko(rows, [single])); ko_p = _embed_rows(_ko(rows, genes))
    pc = PCA(n_components=2, random_state=RANDOM_STATE).fit(base)
    b2, s2, p2 = pc.transform(base), pc.transform(ko_s), pc.transform(ko_p)
    ms, mp = _mcos(base, ko_s), _mcos(base, ko_p)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharex=True, sharey=True)
    for ax, dst, ttl in [(axes[0], s2, f"single-gene KO: {single}  (shift={ms:.4f})"),
                         (axes[1], p2, f"program KO: {prog}  (shift={mp:.4f})")]:
        ax.scatter(b2[:, 0], b2[:, 1], s=20, c="lightgray", zorder=1)
        ax.quiver(b2[:, 0], b2[:, 1], dst[:, 0] - b2[:, 0], dst[:, 1] - b2[:, 1],
                  angles="xy", scale_units="xy", scale=1, color="crimson", width=0.005, alpha=0.85, zorder=2)
        ax.set_title(ttl, fontsize=10); ax.set_xlabel("embedding axis 1")
    axes[0].set_ylabel("embedding axis 2")
    fig.suptitle(f"[{prog}]  arrows = cell displacement | left: single gene (tiny) -> right: whole program (large)  (n={len(cells)})", fontsize=11)
    plt.tight_layout(); plt.show()
    print(f"  [{prog}] shift: single '{single}'={ms:.4f}  vs  program={mp:.4f}  (배율 {mp/ms:.1f}x)" if ms > 0 else f"  [{prog}] single={ms:.4f} program={mp:.4f}")

# 모든 cell type program을 순회 (housekeeping 음성대조 제외)
for _p in MARKER_SETS:
    viz_program(_p)


effect_shift가 큰 program일수록 그 marker set 제거가 세포 표현을 더 크게 흔든다는 뜻입니다. 한 가지 함정을 짚고 갑니다. effect_shift만으로 순위를 매기면, 특정 cell type에 특이적이지 않아도 **넓게, 크게 발현되는 program**이 위로 올라올 수 있습니다. (주의: Geneformer는 raw 발현량이 아니라 `count / gene median` 순위를 쓰므로, GAPDH나 B2M 같은 housekeeping은 median이 커서 오히려 **순위가 낮아집니다** - 섹션 3에서 본 정규화 효과. 따라서 'housekeeping이라 rank 최상위'라는 직관은 Geneformer에는 맞지 않습니다.) 순위(낮음)와 deletion 효과(큼)는 별개의 축입니다 — housekeeping의 effect_shift가 큰 이유는 rank가 높아서가 아니라, 그 세포 안에서 **발현량 자체가 커서** 지우면 표현이 크게 흔들리고(게다가 거의 모든 세포에서 발현돼 그 효과가 폭넓게 나타나기) 때문입니다. 그래서 effect_shift 하나로는 부족하고, 다음 단계에서 **specificity**(한 cell type에 발현이 집중되는 정도)를 함께 봐야 진짜 cell-type-specific program이 드러납니다. effect_shift가 크다고 그 program이 치료 표적이라는 뜻도 아닙니다 - 후보 우선순위화의 단서일 뿐이고, 실제 인과는 held-out perturbation과 functional assay가 필요합니다.

## 11. Target prioritization

여러 단서를 하나의 우선순위 점수로 묶습니다. effect_shift 외에 specificity를 더합니다. specificity(주의: 통계의 민감도·특이도 TN/(TN+FP)가 아니라, 발현이 한 cell type에 집중된 정도)는 program의 marker 발현이 한 cell type에 얼마나 집중되는지로, marker들의 cell type별 평균 발현을 합산한 뒤 최대값을 합으로 나눈 값입니다. 값이 1에 가까울수록 특정 cell type에 특이적이고(cell-type program), 고르게 퍼져 있으면 낮습니다(housekeeping). 각 program의 specificity를 계산합니다.


In [ ]:
def specificity(program):
    accum = None
    for g in program_genes(program):
        s = gene_mean_by_celltype(g)
        if s is None:
            continue
        accum = s.copy() if accum is None else accum.add(s, fill_value=0)
    if accum is None or float(accum.sum()) == 0:
        return 0.0
    return float(accum.max() / accum.sum())

targets["specificity"] = [round(specificity(p), 3) for p in targets["program"]]
display(targets[["program", "effect_shift", "specificity", "literature_support"]])


이제 단서를 묶어 priority score를 만듭니다. effect_shift와 specificity를 0~1로 정규화하고, literature support를 더한 가중합으로 점수를 냅니다. 동시에 failure flag를 답니다. effect를 측정 못 했거나(gene 미검출), specificity가 너무 낮으면 플래그를 답니다. 점수만 보지 말고 플래그를 함께 읽어, 점수가 높아도 신뢰하기 어려운 후보를 가립니다.


> priority_score의 가중치(effect 1.0, specificity 0.8, literature 0.5, flag -0.5)는 이 실습을 위한 **예시 값**이며 최적화된 것이 아닙니다. 또 effect와 specificity는 이 5개 후보 안에서 0~1로 정규화하므로 **점수는 이 후보군 내부의 상대값**입니다 - 후보를 더하거나 빼면 값이 달라질 수 있어 절대 점수처럼 읽으면 안 됩니다.

In [ ]:
def norm01(s):
    s = pd.Series(s, dtype=float)
    return (s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else pd.Series(0.0, index=s.index)

rank_df = targets.copy()
rank_df["eff_norm"] = norm01(rank_df["effect_shift"].fillna(0))
rank_df["spec_norm"] = norm01(rank_df["specificity"])
rank_df["flags"] = [
    "; ".join(
        ([] if r["n_cells"] > 0 and not np.isnan(r["effect_shift"]) else ["not_detected"])
        + ([] if r["specificity"] >= 0.3 else ["low_specificity"])
    ) or "(none)"
    for _, r in rank_df.iterrows()
]
rank_df["n_flags"] = rank_df["flags"].apply(lambda s: 0 if s == "(none)" else s.count(";") + 1)


In [ ]:
rank_df["priority_score"] = (
    rank_df["eff_norm"] + 0.8 * rank_df["spec_norm"] + 0.5 * rank_df["literature_support"]
    - 0.5 * rank_df["n_flags"]
)
rank_df = rank_df.sort_values("priority_score", ascending=False).reset_index(drop=True)
rank_df["rank"] = np.arange(1, len(rank_df) + 1)
display(rank_df[["rank", "program", "effect_shift", "specificity", "literature_support", "flags", "priority_score"]].round(3))


이 표가 이 분석의 핵심 산출물입니다. 맨 위 후보부터 읽되, priority_score 숫자만 보지 말고 그 점수를 만든 단서와 failure flag를 함께 봅니다. effect_shift만 보면 발현량이 큰 housekeeping program이 위로 올라오지만, specificity가 낮아(그리고 low_specificity 플래그가 붙어) 최종 priority에서는 하위로 내려갑니다. effect와 specificity를 함께 써야 cell-type-specific program이 상위로 오는 것, 이것이 이 점수 구성의 핵심입니다. 다시 강조하면, 이 순위는 follow-up 실험을 고르기 위한 우선순위이지 치료 표적이나 임상 유효성의 순위가 아닙니다. priority_score를 therapeutic claim으로 바꾸려면 perturb-seq, functional assay, held-out validation이 필요합니다.


## 12. single-cell foundation model 개요

오늘 Geneformer를 직접 다뤘지만, single-cell foundation model은 입력 방식에 따라 여러 갈래가 있습니다. 모델 카드를 읽을 때 "cell을 어떻게 token으로 바꾸는가"를 먼저 보면 결과 해석의 기준이 잡힙니다. 아래 표로 대표 모델을 정리합니다.


In [ ]:
model_overview = pd.DataFrame([
    {"모델": "scBERT", "입력": "binned expression", "특징": "초기 scFM, efficient attention"},
    {"모델": "Geneformer", "입력": "rank-ordered genes", "특징": "약 30M cells, in silico perturbation"},
    {"모델": "scGPT", "입력": "gene + expression + condition", "특징": "generative, annotation·integration·perturbation"},
    {"모델": "scFoundation / CellFM", "입력": "large-scale corpus", "특징": "scale-up, multi-task transfer"},
])
display(model_overview)


## 13. 정리: 무엇을 만들었고, 무엇을 말하면 안 되는가

이 노트북에서 우리는 PBMC3k의 cell × gene matrix를 전처리하고, rank-value tokenization 개념을 확인하고, 실제 Geneformer로 세포 임베딩을 추출했습니다. 그 위에서 marker annotation, supervised probing, batch integration, in silico perturbation, target prioritization을 수행했습니다. 모든 단계에서 점수는 후보를 정렬하거나 표현을 비교하는 도구였지, 생물학적 인과나 임상적 결론의 증거가 아니었습니다.


### scFM 결과를 읽는 순서

1. **입력**: counts인지 normalized인지, gene이 symbol인지 Ensembl ID인지, metadata(batch·donor) 품질은 어떤지.
2. **tokenization**: 모델이 rank를 쓰는지, gene+value를 쓰는지. 그래서 무엇을 보존하고 무엇을 버리는지.
3. **embedding**: baseline(PCA/HVG)과 비교해 foundation model이 무엇을 더해 주는지.
4. **annotation/integration**: reference coverage, confidence, batch mixing과 cell-state 보존을 함께.
5. **검증**: split(donor/batch/tissue holdout), baseline(scVI/Harmony), claim level.


### perturbation 결과로 말할 수 있는 것과 아직 말하면 안 되는 것

같은 embedding shift라도 어느 수준의 주장까지 허용되는지는 다릅니다. 이 단계 구분을 지키는 것이 결과를 책임 있게 전달하는 핵심입니다.

- **embedding shift**: "모델 안에서 이 gene을 제거하면 해당 cell type 표현에서 멀어진다" — 모델 출력 사실.
- **state hypothesis**: "그 cell state 신호가 약해진다는 후보 가설" — 검증 필요.
- **network hypothesis**: "관련 pathway gene이 함께 변한다는 후보 네트워크" — co-expression은 regulation이 아님.
- **therapeutic claim**: 치료 표적 주장은 독립 perturb-seq와 functional assay 없이는 말하지 않음.


### 자주 생기는 해석 오류

- UMAP 모양이나 cluster 분리를 cell type 정답으로 단정하는 것.
- annotation accuracy가 높으면 marker pseudo-label이 생물학적으로 옳다고 결론짓는 것.
- batch silhouette만 보고 integration이 성공했다고 하는 것(cell-state 보존을 놓침).
- rare cell type의 embedding이나 confidence가 불안정한 것을 무시하는 것.
- perturbation embedding shift를 곧장 therapeutic target으로 연결하는 것.


### 확인 문제

아래 질문에 스스로 답해 보면, 오늘 다룬 개념이 정리됩니다.

1. cell × gene matrix에는 왜 DNA 서열 같은 자연스러운 순서가 없는가? 그것이 tokenizer 설계에 어떤 자유와 가정을 만드는가?
2. Geneformer의 rank-ordering과 scGPT의 gene+expression 입력은 각각 어떤 정보를 보존하고 무엇을 버리는가?
3. cell type annotation accuracy가 높아도 같은 임베딩을 batch 색으로 다시 봐야 하는 이유는 무엇인가?
4. batch integration에서 batch silhouette만 낮추면 안 되고 cell-type silhouette을 함께 봐야 하는 이유는?
5. in silico perturbation의 embedding shift를 therapeutic target claim으로 말하면 안 되는 이유를 claim level 단계로 설명해 보라.
6. PBMC3k 분석을 실제 disease 데이터로 확장할 때, gene vocabulary·reference coverage·batch·donor 중 무엇을 가장 먼저 점검하겠는가?
